In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

import re
import numpy as np
import pandas as pd
from collections import defaultdict
import pickle

import logomaker
import matplotlib.pyplot as plt

In [ ]:
def score_to_pwm(scores):
    bg = 0.25
    log_odds = scores / np.std(scores)
    pwm = np.exp(log_odds)
    pwm /= pwm.sum(axis=1, keepdims=True)
    return pwm

# def score_to_pwm(scores, T=0.01):
#     x = scores / T
#     x = x - x.max(axis=1, keepdims=True)  # 稳定性
#     exp_x = np.exp(x)
#     return exp_x / exp_x.sum(axis=1, keepdims=True)

def pwm_to_meme(pwm, motif_name="motif1", out_file="query.meme"):
    L = pwm.shape[0]

    with open(out_file, "w") as f:
        f.write("MEME version 4\n\n")
        f.write("ALPHABET= ACGT\n\n")
        f.write("strands: + -\n\n")
        f.write("Background letter frequencies\n")
        f.write("A 0.25 C 0.25 G 0.25 T 0.25\n\n")
        f.write(f"MOTIF {motif_name}\n")
        f.write(f"letter-probability matrix: alength= 4 w= {L}\n")

        for row in pwm:
            f.write(" ".join(f"{x:.6f}" for x in row) + "\n")

In [ ]:
bases = ["A", "C", "G", "T"]

### Human HBG1 promoter

In [ ]:
with open("mutagenesis_Omni-DNA-1b_human.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
scores = defaultdict(dict)

for k, v in data.items():
    if not k.startswith("mut_"):
        continue

    # 解析 key: mut_0_C_A
    m = re.match(r"mut_(\d+)_([ACGT])_([ACGT])", k)
    if not m:
        continue

    pos = int(m.group(1))
    ref = m.group(2)
    mut = m.group(3)

    scores[pos][mut] = v["score"]   # 推荐用 score 或 diff
    scores[pos][ref] = 0.0          # wild-type 设为 0

In [ ]:
L = max(scores.keys()) + 1
score_mat = np.zeros((L, 4))

# Ref as max, others are (max - score)
# for i in range(L):
#     for j, b in enumerate(bases):
#         score_mat[i, j] = abs(scores[i].get(b, 0.0))
#     max_s = max(score_mat[i])
#     for j in range(4):
#         s = score_mat[i][j]
#         if s == 0:
#             score_mat[i][j] = max_s
#         else:
#             score_mat[i][j] = max_s - s
# Ref as max, others are 0
for i in range(L):
    for j, b in enumerate(bases):
        score_mat[i, j] = abs(scores[i].get(b, 0.0))
    max_s = max(score_mat[i])
    for j in range(4):
        s = score_mat[i][j]
        if s == 0:
            score_mat[i][j] = max_s
        else:
            score_mat[i][j] = 0.0
# Ref as max, others are 0
# all_scores = [[]]
# bases = ["A", "C", "G", "T"]
# for i in range(L):
#     for j, b in enumerate(bases):
#         score_mat[i, j] = abs(scores[i].get(b, 0.0))
#     max_s = max(score_mat[i])
#     for j in range(4):
#         s = score_mat[i][j]
#         if s == 0:
#             all_scores[0].append((bases[j], max_s))
#         else:
#             all_scores[0].append((bases[j], 0.0))

In [ ]:
matrix_data = pd.DataFrame(score_mat, columns=bases)

logo = logomaker.Logo(matrix_data, shade_below=.5, fade_below=.5, show_spines=False)
plt.savefig("HBG1.pdf")
plt.show()

In [ ]:
importance = score_mat.std(axis=1)

# 假设 motif 区间是 [start:end]
# motif_scores = score_mat[101:108]
# motif_scores = score_mat[128:136]
motif_scores = score_mat[155:166]

# score -> pwm
pwm = score_to_pwm(motif_scores)

In [ ]:
bases = np.array(["A", "C", "G", "T"])
consensus = "".join(bases[pwm.argmax(axis=1)])
print(consensus)

In [ ]:
pwm_to_meme(
    pwm,
    motif_name="Omni-DNA-1b-HBG1-3",
    out_file="HBG1-3.meme"
)

### Arabidopsis Try intron enhancer

In [ ]:
with open("mutagenesis_plant-dnamamba-BPE-open_chromatin_plant.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
scores = defaultdict(dict)

for k, v in data.items():
    if not k.startswith("mut_"):
        continue

    # 解析 key: mut_0_C_A
    m = re.match(r"mut_(\d+)_([ACGT])_([ACGT])", k)
    if not m:
        continue

    pos = int(m.group(1))
    ref = m.group(2)
    mut = m.group(3)

    scores[pos][mut] = v["score"]   # 推荐用 score 或 diff
    scores[pos][ref] = 0.0          # wild-type 设为 0

In [ ]:
L = max(scores.keys()) + 1
score_mat = np.zeros((L, 4))

for i in range(L):
    for j, b in enumerate(bases):
        score_mat[i, j] = abs(scores[i].get(b, 0.0))
    max_s = max(score_mat[i])
    for j in range(4):
        s = score_mat[i][j]
        if s == 0:
            score_mat[i][j] = max_s
        else:
            score_mat[i][j] = 0.0

In [ ]:
matrix_data = pd.DataFrame(score_mat, columns=bases)
logo = logomaker.Logo(matrix_data, shade_below=.5, fade_below=.5, show_spines=False)
plt.savefig("TryIn2En.pdf")
plt.show()

In [ ]:
score_mat = np.zeros((L, 4))
# Ref as max, others are (max - score)
for i in range(L):
    for j, b in enumerate(bases):
        score_mat[i, j] = abs(scores[i].get(b, 0.0))
    max_s = max(score_mat[i])
    for j in range(4):
        s = score_mat[i][j]
        if s == 0:
            score_mat[i][j] = max_s
        else:
            score_mat[i][j] = max_s - s

In [ ]:
importance = score_mat.std(axis=1)

# 假设 motif 区间是 [start:end]
motif_scores = score_mat[10:27]
motif_scores = score_mat[114:123]
motif_scores = score_mat[131:140]

# score -> pwm
pwm = score_to_pwm(motif_scores)

# pwm -> meme
pwm_to_meme(
    pwm,
    motif_name="PlantCAD2-Large-Try-3",
    out_file="Try-3.meme"
)

### Yeast artifical promoter (P_GPD)

In [ ]:
with open("mutagenesis_Evo2_7b_yeast.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
scores = defaultdict(dict)

for k, v in data.items():
    if not k.startswith("mut_"):
        continue

    # 解析 key: mut_0_C_A
    m = re.match(r"mut_(\d+)_([ACGT])_([ACGT])", k)
    if not m:
        continue

    pos = int(m.group(1))
    ref = m.group(2)
    mut = m.group(3)

    scores[pos][mut] = v["score"]   # 推荐用 score 或 diff
    scores[pos][ref] = 0.0          # wild-type 设为 0

In [ ]:
L = max(scores.keys()) + 1
score_mat = np.zeros((L, 4))

for i in range(L):
    for j, b in enumerate(bases):
        score_mat[i, j] = abs(scores[i].get(b, 0.0))
    max_s = max(score_mat[i])
    for j in range(4):
        s = score_mat[i][j]
        if s == 0:
            score_mat[i][j] = max_s
        else:
            score_mat[i][j] = 0.0

In [ ]:
matrix_data = pd.DataFrame(score_mat, columns=bases)

logo = logomaker.Logo(matrix_data, shade_below=.5, fade_below=.5, show_spines=False)
plt.savefig("Yeast.pdf")
plt.show()

In [ ]:
score_mat = np.zeros((L, 4))
# Ref as max, others are (max - score)
for i in range(L):
    for j, b in enumerate(bases):
        score_mat[i, j] = abs(scores[i].get(b, 0.0))
    max_s = max(score_mat[i])
    for j in range(4):
        s = score_mat[i][j]
        if s == 0:
            score_mat[i][j] = max_s
        else:
            score_mat[i][j] = max_s - s

In [ ]:
importance = score_mat.std(axis=1)

# 假设 motif 区间是 [start:end]
# motif_scores = score_mat[56:69]
# motif_scores = score_mat[92:100]
motif_scores = score_mat[195:203]

# score -> pwm
pwm = score_to_pwm(motif_scores)

# pwm -> meme
pwm_to_meme(
    pwm,
    motif_name="Evo2-7b-Yeast-3",
    out_file="Yeast-3.meme"
)